# Lezione 44 — Impacchettare e distribuire un adapter

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 40 (LoRA da zero).
>
> Obiettivo pratico unico: **salvare e ricaricare** solo l'adapter LoRA (pochi
> KB) invece dell'intero modello, verificando che l'output sia identico — chiude
> la Fase 6.

## Teoria essenziale

Il vantaggio operativo di LoRA: l'adattamento vive in due piccole matrici
$A,B$. Non devi distribuire un modello da gigabyte per ogni compito — distribuisci
un **adapter** da pochi KB, che si applica sopra la stessa base condivisa. Piu'
compiti = piu' adapter intercambiabili, un solo modello base.

Salvare l'adapter = salvare $A$, $B$ e i metadati (rango, scala). Ricaricarlo =
rimetterli sopra la base congelata e ottenere **lo stesso** comportamento.

In [1]:
import numpy as np
from pathlib import Path

rng = np.random.default_rng(44)
d, k, r = 128, 64, 4

W0 = rng.normal(size=(d, k)).astype(np.float32) * 0.3      # base condivisa
A = rng.normal(size=(r, k)).astype(np.float32) * 0.1
B = rng.normal(size=(d, r)).astype(np.float32) * 0.1
scala = 1.0 / r

def forward(x, W0, B, A, scala):
    return x @ W0 + scala * (x @ B @ A)

x = rng.normal(size=(5, d)).astype(np.float32)
out_originale = forward(x, W0, B, A, scala)

In [2]:
cartella = Path("adapter_memoria")
cartella.mkdir(exist_ok=True)
percorso = cartella / "adapter.npz"

# salvo SOLO l'adapter + metadati, non la base
np.savez(percorso, A=A, B=B, rank=np.int32(r), scala=np.float32(scala))

byte_adapter = percorso.stat().st_size
byte_base = W0.nbytes
print(f"adapter su disco: {byte_adapter} byte")
print(f"base W0 (non distribuita): {byte_base} byte")
print(f"l'adapter e' circa {byte_base/byte_adapter:.1f}x piu' piccolo della sola base")

adapter su disco: 4052 byte
base W0 (non distribuita): 32768 byte
l'adapter e' circa 8.1x piu' piccolo della sola base


In [3]:
# ricarico l'adapter e lo applico alla STESSA base
dati = np.load(percorso)
A2, B2 = dati["A"], dati["B"]
scala2 = float(dati["scala"])
out_ricaricato = forward(x, W0, B2, A2, scala2)

# controllo di non-regressione: comportamento identico dopo il round-trip
assert np.allclose(out_originale, out_ricaricato, atol=1e-6)
assert int(dati["rank"]) == r
print("round-trip salva/ricarica: output identico  ✓")
print("rango ripristinato dai metadati:", int(dati["rank"]))

round-trip salva/ricarica: output identico  ✓
rango ripristinato dai metadati: 4


## Il progetto: Memory AI Lab, passo 44

Impacchetto salvataggio e caricamento come due funzioni pulite, con i metadati
necessari a riapplicare l'adapter senza indovinare iperparametri. E' l'ultimo
mattone della Fase 6: il progetto puo' ora tenere piu' adapter (uno per tipo di
adattamento) sopra un'unica base.

In [4]:
def salva_adapter(percorso, A, B, scala):
    np.savez(percorso, A=A, B=B, scala=np.float32(scala), rank=np.int32(A.shape[0]))

def carica_adapter(percorso):
    d = np.load(percorso)
    return {"A": d["A"], "B": d["B"], "scala": float(d["scala"]), "rank": int(d["rank"])}

p = cartella / "adapter2.npz"
salva_adapter(p, A, B, scala)
ad = carica_adapter(p)
assert ad["A"].shape == (r, k) and ad["B"].shape == (d, r)
assert np.allclose(forward(x, W0, ad["B"], ad["A"], ad["scala"]), out_originale)
print("adapter salvato e ricaricato correttamente; rank:", ad["rank"])

# pulizia dei file temporanei creati dalla lezione
for f in cartella.glob("*.npz"):
    f.unlink()
cartella.rmdir()
print("file temporanei rimossi")

adapter salvato e ricaricato correttamente; rank: 4
file temporanei rimossi


## Riepilogo (max 8 punti)

1. L'adattamento LoRA vive in due piccole matrici $A,B$.
2. Distribuisci un **adapter** (pochi KB), non un modello da gigabyte.
3. Una sola base condivisa + molti adapter intercambiabili.
4. Salvare = $A$, $B$ + metadati (rango, scala).
5. Ricaricare sulla stessa base da' comportamento **identico** (verificato).
6. I metadati evitano di indovinare gli iperparametri.
7. L'adapter e' molto piu' piccolo della sola base.
8. Chiude la Fase 6: piu' adapter su un'unica base.

## Quiz

1. Perche' non serve salvare $W_0$ con l'adapter?
2. Cosa serve nei metadati per riapplicare correttamente l'adapter?
3. Qual e' il vantaggio operativo di avere piu' adapter su una base?

*(Risposte: 1. la base e' condivisa e congelata, uguale per tutti gli adapter;
2. rango e fattore di scala (oltre alle matrici); 3. si serve un solo modello
base in memoria e si scambiano adapter leggeri per compito.)*